### Rag PipeLine -> Data ingestion to vector DB PipeLine

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
# Read all the pdf inside directory

def process_all_pdfs(pdf_directory):

    all_documents=[]
    pdf_dir=Path(pdf_directory)

    # find all pdf file recursively inside the directory
    # ** mean search all folder and sub folder recursively
    # pdf_files is now have list of all pdf in pdf_directory and sub directory of pdf_directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            # Load the PDF file using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:

            #    for adding extra information in meta data like here we add file name and type of file

               doc.metadata['source_file'] = pdf_file.name 
               doc.metadata['file_type']='pdf'

            # extend add the add pdf document in one list
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files in ../data
Processing ../data/file-sample_150kB.pdf...
Loaded 4 pages
Processing ../data/iso27001.pdf...
Loaded 26 pages
Total documents loaded: 30


In [8]:
all_pdf_documents

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit

In [ ]:
# now we convert our documents into chunks

# chunk_size is the size of each chunk in number of characters
# chunks_overlap is the number of characters that overlap between chunks
# overlap chunks example
# Chunk 1:    1 -------- 1000
# Chunk 2:         801 -------- 1800
# Chunk 3:               1601 ------- 2500
def split_documents(documents, chunk_size=1000, chunks_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunks_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_documents=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_documents)} chunks")

    # show example of chunks

    if split_documents:
        print("// Example of a chunk:")
        print(f'content: {split_documents[0].page_content[:200]}...')  # Print first 200 characters of the first chunk
        print(f'chunk metadata: {split_documents[0].metadata}')  # Print metadata of the first chunk

    return split_documents


In [13]:
chunks=split_documents(all_pdf_documents)
chunks

split 30 documents into 85 chunks
// Example of a chunk:
content: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nulla...
chunk metadata: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit

### Embeding and vector Store Db

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbedingManager:

    # model is used to convert text into vectors
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Load successfully. Embeding dimention: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def genrate_embedding(self, texts: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model is not loaded. Please call _load_model() first.")
            
        print(f"Generating embedding for {len(texts)} characters")  # Print first 50 characters of the text
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding shape: {embedding.shape}")  # Print the shape of the generated embedding
        return embedding

    # def get_embedding_dimension(self) -> int:
    #     if not self.model:
    #         raise ValueError("Model is not loaded. Please call _load_model() first.")
    #     return self.model.get_sentence_embedding_dimension()


# initialize the EmbedingManager with the desired model
embedding_manager = EmbedingManager(model_name="all-MiniLM-L6-v2")
embedding_manager
       

KeyboardInterrupt: 